# 03 — Train the centralised classifier

This is the baseline the whole thesis is measured against. One machine holds all
the data and trains one model, which is what the federated runs in notebooks 06
and 07 have to match without ever pooling the images.

Everything is written out here rather than imported: the dataset, the sampler, the
model head, the training loop, the metrics and the figures. The reason is that
several of the decisions in this loop look wrong until you read why they are
there, and a comment three files away does not get read. The MPS section in
particular cost this project a week of believing the Apple GPU was broken.

**What this notebook needs**

`dataset/multi_subtype_80mm/` must exist, with `train.csv`, `val.csv`, `test.csv`
and the `images/` tree. Build it with notebook 02.

**What it produces**

A numbered run folder holding the two checkpoints, the per-epoch log, the
per-patient predictions, every metric and every figure. Exact paths are listed in
the last cell.

**One number to keep in mind before reading any result.** Two runs of a
byte-identical configuration, differing only in the seed, were measured 0.067
macro-AUC apart on this task. That is the noise floor. One seed is not a result,
and any difference smaller than 0.067 is not a difference.

## Configuration

Everything is here. No path and no hyperparameter appears anywhere below this
cell, so this is the only place to change an experiment.

The values below are the measured configuration, not defaults. They are what the
reported centralised baseline used and what every federated client uses, and that
matching is the entire basis of the comparison. If the federated arm trained with
different regularisation, the gap being measured would be the regularisation
rather than the federation.

One environment variable is set before torch is imported, and it has to be: the
MPS fallback flag only takes effect if it is set first.

In [ ]:
import os

# MUST be set BEFORE torch is imported. An operation Apple's Metal backend has not
# implemented falls back to the CPU instead of raising. Setting it after the import
# silently does nothing.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# os.environ["BREAST_FORCE_CPU"] = "1"        # ignore every accelerator, run on CPU
# os.environ["BREAST_CACHE_IMAGES"] = "1"     # decode all 16,378 PNGs into RAM (2.5 GB).
#                                             # Throughput only, cannot change a result.
#                                             # Worth it on Apple silicon, where PNG
#                                             # decoding leaves the GPU 88% idle.
# os.environ["BREAST_DATALOADER_WORKERS"] = "4"   # override the worker count below

from pathlib import Path

# --------------------------------------------------------------------------- #
# PATHS
# --------------------------------------------------------------------------- #

# Repository root. The notebook lives in notebooks/, so the parent is the root.
REPO_ROOT = Path.cwd().parent

# INPUT, must exist. Built by notebook 02. Expect train.csv, val.csv, test.csv,
# metadata.csv, config.json and an images/ tree with one folder per patient.
DATASET_DIR = REPO_ROOT / "dataset" / "multi_subtype_80mm"
IMAGES_DIR = DATASET_DIR / "images"

# OUTPUT root. Each run gets its own numbered subfolder inside this, named
# test_NNN_<model>_<task>. The number is one above the highest already present,
# and gaps are never reused, so a deleted run cannot have its number silently
# taken by a later one.
RESULTS_ROOT = REPO_ROOT / "results" / "classifier"

# --------------------------------------------------------------------------- #
# TASK
# --------------------------------------------------------------------------- #

TASK = "subtype"
CLASS_NAMES = ("HRposHER2neg", "TripleNeg", "HER2pos")
NUM_CLASSES = len(CLASS_NAMES)

# --------------------------------------------------------------------------- #
# MODEL
# --------------------------------------------------------------------------- #

MODEL = "resnet18"              # 11.2M, the measured winner on this task
# MODEL = "resnet34"            # 21.3M
# MODEL = "resnet50"            # 23.5M, the reference in the comparable literature
# MODEL = "efficientnet_b0"     #  4.0M, lighter and faster to train
# MODEL = "efficientnet_b3"     # 10.7M
# MODEL = "convnext_tiny"       # 27.8M
# MODEL = "convnext_small"      # 49.5M
# MODEL = "mobilenet_v3_large"  #  4.2M
# MODEL = "densenet121"         #  7.0M
# MODEL = "vit_b_16"            # 85.8M, torchvision ViT, needs the lower LR below
# MODEL = "swin_t"              # 27.5M, also needs the lower LR
#
# Architecture does not decide this task. From 1.5M to 87.6M parameters, CNNs and
# transformers alike, every one lands in the same 0.55 to 0.63 macro-AUC band. The
# ceiling is signal, not capacity, so the cheapest adequate backbone is the right one.

PRETRAINED = True               # ImageNet weights. False trains from scratch and
                                # loses about 0.05 macro-AUC on this data volume.

# --------------------------------------------------------------------------- #
# OPTIMISATION
# --------------------------------------------------------------------------- #

SEED = 42                       # run at least two, the noise floor is 0.067

OPTIMIZER = "adamw"
# OPTIMIZER = "sgd"             # momentum 0.9 is applied automatically

LEARNING_RATE = 1e-4            # measured best
# LEARNING_RATE = 3e-5          # measured 0.6473. Use this for vit_* and swin_*,
#                               # which diverge at the rate swept for a CNN
# LEARNING_RATE = 3e-4          # measured worst, 0.5728

WEIGHT_DECAY = 5e-4
SCHEDULER = "cosine"            # the federated clients evaluate cosine in closed
                                # form per round, so cosine keeps the two comparable
# SCHEDULER = "plateau"         # halves the rate after 5 stale epochs
# SCHEDULER = "none"            # constant rate

BATCH_SIZE = 24                 # measured best
# BATCH_SIZE = 8                # measured 0.6264
# BATCH_SIZE = 64               # measured 0.6238

EPOCHS = 30                     # matches the federated budget of 30 rounds x 1 local epoch
EARLY_STOPPING_PATIENCE = 0     # 0 disables it. The federated campaign used 0, so
                                # the baseline uses 0 to keep the budgets identical.
# EARLY_STOPPING_PATIENCE = 30  # stop after 30 epochs without improvement

MIXED_PRECISION = True          # applied on CUDA only. On MPS it measured 13% SLOWER
                                # than fp32, and GradScaler is CUDA-specific.

# --------------------------------------------------------------------------- #
# REGULARISATION
# --------------------------------------------------------------------------- #

DROPOUT = 0.5                   # 0.0 disables it and changes the head layout
LABEL_SMOOTHING = 0.1           # 0.0 disables it
CLASS_WEIGHTED_LOSS = True      # inverse frequency, counted per PATIENT not per slice
FREEZE_UNTIL = "layer3"         # freezes conv1 + bn1 + layer1 + layer2
# FREEZE_UNTIL = "none"         # train the whole network
# FREEZE_UNTIL = "layer4"       # 25% of parameters, the honest freezing test, never run
FREEZE_BN = False               # True also pins the BatchNorm running statistics
AUGMENTATION = "default"
# AUGMENTATION = "half"         # same amplitudes, applied 50% of the time. MEASURED
#                               # WORSE: the train/test gap tripled, 0.135 to 0.512
# AUGMENTATION = "none"         # no augmentation at all

# --------------------------------------------------------------------------- #
# BATCHING AND EVALUATION
# --------------------------------------------------------------------------- #

IMAGE_SIZE = 224                # the backbones' native size
MAX_SLICES_PER_PATIENT_PER_BATCH = 1    # 0 or None removes the constraint
NUM_WORKERS = 8                 # forced to 0 on macOS, see the loader cell
AGGREGATION = "mean"            # how slice probabilities become one patient prediction
# AGGREGATION = "median"        # what the authors report as best for binary HER2
MONITOR_METRIC = "auc"          # what selects the saved checkpoint. NEVER a training metric
# MONITOR_METRIC = "balanced_accuracy"
# MONITOR_METRIC = "macro_f1"

CONFIG = dict(
    task=TASK, class_names=list(CLASS_NAMES), num_classes=NUM_CLASSES,
    model=MODEL, pretrained=PRETRAINED, seed=SEED, optimizer=OPTIMIZER,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, scheduler=SCHEDULER,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    mixed_precision=MIXED_PRECISION, dropout=DROPOUT,
    label_smoothing=LABEL_SMOOTHING, class_weighted_loss=CLASS_WEIGHTED_LOSS,
    freeze_until=FREEZE_UNTIL, freeze_bn=FREEZE_BN, augmentation=AUGMENTATION,
    image_size=IMAGE_SIZE,
    max_slices_per_patient_per_batch=MAX_SLICES_PER_PATIENT_PER_BATCH,
    num_workers=NUM_WORKERS, aggregation=AGGREGATION,
    monitor_metric=MONITOR_METRIC, dataset=str(DATASET_DIR),
)

assert DATASET_DIR.is_dir(), f"{DATASET_DIR} missing — run notebook 02 first"
print(f"dataset  {DATASET_DIR}")
print(f"results  {RESULTS_ROOT}")
print(f"model    {MODEL}, {EPOCHS} epochs, batch {BATCH_SIZE}, seed {SEED}")

## Imports and the device

`get_device` picks CUDA, then Apple MPS, then CPU. The long comment on it is the
most expensive thing in this notebook and it is worth reading before trusting any
result produced on a Mac.

The short version: MPS was banned from this project for months on the strength of
a training accuracy of 0.90 in the first epoch, on a three-class task whose
trivial baseline is 0.51. That is not a diverged model, it is corrupted input.
Three separate defects were involved and none of them was in MPS. They are fixed
in the cells further down, each at the point where it bites.

In [ ]:
import json
import platform
import random
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, replace

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})
PALETTE = ["#4c78a8", "#e45756", "#54a24b", "#f58518", "#b279a2"]


def get_device(allow_mps=True):
    """The best available accelerator: CUDA, then Apple MPS, then CPU."""
    if os.environ.get("BREAST_FORCE_CPU") == "1":
        return torch.device("cpu")
    if torch.cuda.is_available():
        return torch.device("cuda")
    if allow_mps and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def describe_device(device=None):
    device = device or get_device()
    if device.type == "cuda":
        name = torch.cuda.get_device_name(0)
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        return f"cuda — {name}, {total:.1f} GB"
    if device.type == "mps":
        return f"mps — Apple {platform.machine()} GPU (Metal Performance Shaders)"
    return f"cpu — {platform.processor() or platform.machine()}"


DEVICE = get_device()
# AMP on CUDA only. On MPS it was measured 13% slower than fp32 and GradScaler is
# CUDA-specific. Throughput only, the maths is identical either way.
USE_AMP = MIXED_PRECISION and DEVICE.type == "cuda"

print(f"torch  {torch.__version__}")
print(f"device {describe_device(DEVICE)}")
print(f"amp    {USE_AMP}")

## Seeding

This pins what can be pinned, and it is worth being precise about what it does not
pin: cuDNN kernel selection, mixed precision, and DataLoader worker ordering. Two
runs of a byte-identical configuration differing only in the seed were measured
0.067 macro-AUC apart here, and two runs of the same seed on different GPUs
diverged as well.

That is why the campaign quotes a noise floor and why nothing in the thesis rests
on a single run.

In [ ]:
def set_seed(seed):
    """Pin what can be pinned. See the markdown above for what this does not pin."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"seeded with {SEED}")

## Augmentation

MRI-appropriate, and what is deliberately absent matters as much as what is there.

MixUp and CutMix at image level are out because they blend lesions from different
patients, and the diagnostic signal in MRI is small, local and low-contrast. An
aggressive vertical flip is out because a cranio-caudal flip produces anatomy that
does not exist; left to right is fine, since it looks like the contralateral
breast. Strong elastic deformation is out because it alters tumour morphology,
which is precisely the feature being classified.

One implementation detail worth noticing: rotation, zoom and translation are drawn
independently but composed into a single affine transform. Three successive
`grid_sample` calls would blur the image three times over.

The `half` profile below is kept because it was measured, not because it is
recommended. Same amplitudes applied half the time took training accuracy from
0.57 to 0.99 and tripled the train/test gap.

In [ ]:
@dataclass(frozen=True)
class AugmentConfig:
    enabled: bool = True

    horizontal_flip: float = 0.5
    vertical_flip: float = 0.0          # a cranio-caudal flip is not anatomy

    rotation_degrees: float = 15.0
    rotation_probability: float = 1.0
    scale_range: tuple = (0.9, 1.1)
    scale_probability: float = 1.0
    translate_fraction: float = 0.08
    translate_probability: float = 1.0

    brightness_range: tuple = (0.8, 1.2)
    brightness_probability: float = 1.0

    gaussian_noise_std: tuple = (0.005, 0.03)
    noise_probability: float = 0.25

    cutout_probability: float = 0.0


PROFILES = {
    "default": AugmentConfig(),
    # Same amplitudes, half the time. MEASURED WORSE, kept for the record.
    "half": replace(AugmentConfig(), rotation_probability=0.5, scale_probability=0.5,
                    translate_probability=0.5, brightness_probability=0.5),
    "none": replace(AugmentConfig(), enabled=False),
}


def apply_augment(img, aug):
    """Augment one image, shape (3, H, W), values in [0, 1]."""
    if random.random() < aug.horizontal_flip:
        img = torch.flip(img, dims=[2])
    if random.random() < aug.vertical_flip:
        img = torch.flip(img, dims=[1])

    do_rot = aug.rotation_degrees > 0 and random.random() < aug.rotation_probability
    do_scale = aug.scale_range != (1.0, 1.0) and random.random() < aug.scale_probability
    do_trans = aug.translate_fraction > 0 and random.random() < aug.translate_probability

    # One affine transform, not three. Three grid_sample calls blur three times.
    if do_rot or do_scale or do_trans:
        angle = np.deg2rad(random.uniform(-aug.rotation_degrees,
                                          aug.rotation_degrees)) if do_rot else 0.0
        scale = random.uniform(*aug.scale_range) if do_scale else 1.0
        tx = random.uniform(-aug.translate_fraction, aug.translate_fraction) if do_trans else 0.0
        ty = random.uniform(-aug.translate_fraction, aug.translate_fraction) if do_trans else 0.0
        cos_a, sin_a = np.cos(angle) / scale, np.sin(angle) / scale
        theta = img.new_tensor([[cos_a, -sin_a, tx], [sin_a, cos_a, ty]]).unsqueeze(0)
        grid = F.affine_grid(theta, img.unsqueeze(0).shape, align_corners=False)
        img = F.grid_sample(img.unsqueeze(0), grid, mode="bilinear",
                            padding_mode="zeros", align_corners=False).squeeze(0)

    if aug.brightness_range != (1.0, 1.0) and random.random() < aug.brightness_probability:
        img = torch.clamp(img * random.uniform(*aug.brightness_range), 0, 1)
    if aug.gaussian_noise_std and random.random() < aug.noise_probability:
        sigma = random.uniform(*aug.gaussian_noise_std)
        img = torch.clamp(img + torch.randn_like(img) * sigma, 0, 1)

    if aug.cutout_probability > 0 and random.random() < aug.cutout_probability:
        _, h, w = img.shape
        side = int(min(h, w) * random.uniform(0.1, 0.3))
        if side > 0:
            top, left = random.randint(0, h - side), random.randint(0, w - side)
            img = img.clone()
            img[:, top:top + side, left:left + side] = 0.0
    return img


AUGMENT = PROFILES[AUGMENTATION]
print(f"augmentation profile: {AUGMENTATION}, enabled={AUGMENT.enabled}")

## The dataset

One PNG per item, and the patient id travels with it because every metric in this
project is computed per patient.

Two details. The ImageNet normalisation is applied last, after augmentation, so
that brightness and noise operate in [0, 1] where clamping to [0, 1] means
something. And the optional in-RAM cache decodes every image once at construction
into one contiguous uint8 array. That is a throughput change and cannot alter a
result, since the cached array is exactly what `np.asarray` returned. It exists
because on Apple silicon PNG decoding dominates the step: the GPU was measured at
12% utilisation against 167% CPU without it.

The cache is built with threads rather than processes on purpose. PIL releases the
GIL during decode, so threads parallelise it, and the result stays in one address
space that DataLoader workers inherit through `fork` without copying it.

In [ ]:
# The backbones were pretrained with these statistics.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class SliceDataset(Dataset):
    """One PNG per item. Returns (image, label, row_position)."""

    def __init__(self, rows, images_dir, image_size, augment=None, cache=False):
        self.rows = rows.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.image_size = image_size
        self.augment = augment
        self._cache = self._build_cache() if cache else None

    def _load_uint8(self, filename):
        img = Image.open(self.images_dir / filename).convert("RGB")
        if img.size != (self.image_size, self.image_size):
            img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        return np.asarray(img, dtype=np.uint8)

    def _build_cache(self):
        n, s = len(self.rows), self.image_size
        buf = np.empty((n, s, s, 3), dtype=np.uint8)
        names = self.rows.filename.tolist()

        def fill(i):
            buf[i] = self._load_uint8(names[i])

        with ThreadPoolExecutor(max_workers=min(16, (os.cpu_count() or 4) * 2)) as ex:
            list(ex.map(fill, range(n)))
        print(f"  cached {n:,} images in RAM ({buf.nbytes / 1e9:.2f} GB)", flush=True)
        return buf

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows.iloc[i]
        arr = (self._cache[i] if self._cache is not None
               else self._load_uint8(r.filename))
        x = torch.from_numpy(arr.astype(np.float32) / 255.0).permute(2, 0, 1)
        if self.augment is not None and self.augment.enabled:
            x = apply_augment(x, self.augment)
        # ImageNet normalisation LAST, after augmentation.
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        return x, int(r.label), i


print("SliceDataset defined")

## The batch sampler

This is the least obvious piece of the whole loop and it matters more than it
looks.

Plain shuffling draws 24 indices from thousands of slices belonging to hundreds of
patients, so a batch routinely contains several near-duplicate slices of the same
tumour. Those produce almost the same gradient, which quietly removes the
stochastic noise that makes SGD generalise in the first place.

Every slice is still seen exactly once per epoch. This is not subsampling, only
batch composition changes.

One bug worth recording, because it is easy to reintroduce. The `used` counter
resets when a batch is emitted, not on every pass through the queue. Resetting per
pass let a patient enter the same batch twice once few patients remained, which is
precisely what this sampler exists to prevent.

In [ ]:
class PatientBatchSampler(Sampler):
    """Batches holding at most `max_per_patient` slices of any one patient."""

    def __init__(self, pids, batch_size, max_per_patient=1, seed=0):
        self.batch_size = batch_size
        self.max_per_patient = max(1, max_per_patient)
        self.seed = seed
        self.epoch = 0
        self.groups = {}
        for i, pid in enumerate(pids):
            self.groups.setdefault(pid, []).append(i)
        self.n_samples = len(pids)

    def set_epoch(self, epoch):
        """Reshuffle differently each epoch. Identical batch composition every
        epoch would itself be a form of overfitting."""
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed * 100003 + self.epoch)
        queues = {p: rng.sample(idx, len(idx)) for p, idx in self.groups.items()}
        batch, used = [], {}
        while queues:
            available = list(queues)
            rng.shuffle(available)
            added = 0
            for pid in available:
                if len(batch) >= self.batch_size:
                    break
                if used.get(pid, 0) >= self.max_per_patient:
                    continue
                batch.append(queues[pid].pop())
                used[pid] = used.get(pid, 0) + 1
                added += 1
                if not queues[pid]:
                    del queues[pid]
            # `used` resets when the batch is EMITTED, not on every pass.
            if len(batch) >= self.batch_size or added == 0 or not queues:
                if batch:
                    yield batch
                batch, used = [], {}
        if batch:
            yield batch

    def __len__(self):
        return (self.n_samples + self.batch_size - 1) // self.batch_size


print("PatientBatchSampler defined")

## Loaders, class weights and the trivial baseline

The loaders return the dataframes alongside them, because every downstream step
(class weights, patient-level aggregation, per-cohort reporting) must score exactly
the rows the loader fed and not a re-read of the CSV.

Worker count is forced to 0 on macOS. DataLoader workers use `spawn` there and
reliably hang when the parent holds an MPS context. Set `BREAST_DATALOADER_WORKERS`
in the config cell to override, and if you do, turn the image cache on as well:
workers then only run augmentation on arrays they inherit through `fork` and never
rebuild the MPS context that made `spawn` hang.

The class weights are counted per patient rather than per slice. Counting slices
conflates two different things, how many patients carry a class and how large their
tumours are, so a subtype whose tumours happen to be large would get a small weight
simply for contributing more images.

The trivial baseline is the majority-class rate among patients. Accuracy is
meaningless without it, and it is not a constant: 0.404 on I-SPY2 alone against
0.511 pooled.

In [ ]:
def effective_num_workers(requested):
    """Workers for the DataLoader. Throughput only, cannot change a result."""
    override = os.environ.get("BREAST_DATALOADER_WORKERS")
    if override is not None:
        return max(0, int(override))
    return 0 if platform.system() == "Darwin" else requested


def make_loaders(dataset_dir, images_dir):
    """train / val / test loaders, and the frames behind them."""
    workers = effective_num_workers(NUM_WORKERS)
    cache = os.environ.get("BREAST_CACHE_IMAGES") == "1"

    # `fork` rather than the macOS default `spawn`: workers then inherit the image
    # cache instead of re-pickling 2.5 GB each, and never rebuild the MPS context.
    if workers > 0:
        import multiprocessing as mp
        try:
            mp.set_start_method("fork", force=True)
        except RuntimeError:
            pass

    loaders, frames = {}, {}
    for split in ("train", "val", "test"):
        csv_path = Path(dataset_dir) / f"{split}.csv"
        if not csv_path.is_file():
            continue
        rows = pd.read_csv(csv_path)
        frames[split] = rows
        ds = SliceDataset(rows, images_dir, IMAGE_SIZE,
                          AUGMENT if split == "train" else None, cache=cache)
        common = dict(num_workers=workers, pin_memory=torch.cuda.is_available(),
                      persistent_workers=workers > 0)
        if split == "train":
            sampler = PatientBatchSampler(rows.pid.tolist(), BATCH_SIZE,
                                          MAX_SLICES_PER_PATIENT_PER_BATCH,
                                          seed=SEED)
            loaders[split] = DataLoader(ds, batch_sampler=sampler, **common)
        else:
            # No shuffling and no dropping: evaluation must see every row once.
            loaders[split] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                                        drop_last=False, **common)
    return loaders, frames


def class_weights(rows, num_classes, device):
    """Inverse-frequency weights counted per PATIENT, not per slice."""
    per_patient = rows.groupby("pid").label.first()
    counts = np.bincount(per_patient.values, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = len(per_patient) / (num_classes * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)


def trivial_baseline(rows):
    """Majority-class rate among PATIENTS."""
    per_patient = rows.drop_duplicates("pid").label
    return float(per_patient.value_counts().max() / len(per_patient)) if len(per_patient) else float("nan")


loaders, frames = make_loaders(DATASET_DIR, IMAGES_DIR)
for split, rows in frames.items():
    print(f"  {split:<5} {len(rows):>7,} slices / {rows.pid.nunique():>5} patients"
          f"   trivial baseline {trivial_baseline(rows):.4f}")

## The model and its head

One name in, one ready-to-train network out. The head deserves an explanation
because getting it wrong cost this project seven unloadable checkpoints.

Every backbone ends with exactly one `Dropout(p)` feeding the final `Linear`, where
`p` is the configured dropout. That uniformity is deliberate: `dropout` is written
into every `results.json`, so a build that honoured it for some backbones and
ignored it for others would report hyperparameters that do not describe the trained
model.

The rule that keeps checkpoints loadable is never to insert into an existing
`Sequential`. Inserting shifts every index after it, so a checkpoint saved by one
build fails to load into another. The final `Linear` is therefore only ever
replaced in place, and where a backbone already ships a `Dropout` feeding it
(efficientnet, mobilenet) that `Dropout` is retuned rather than stacked with a
second one.

The failure mode this avoids is nasty: loading with `strict=False` succeeds, leaves
the classifier at random init, and produces a near-chance score that reads as a
result rather than as a bug. Hence `strict=True` everywhere.

In [ ]:
def new_head(in_features, num_classes, dropout):
    """Dropout(p) -> Linear, or a bare Linear when p is 0."""
    head = nn.Linear(in_features, num_classes)
    return nn.Sequential(nn.Dropout(dropout), head) if dropout > 0 else head


def retune_dropout_before(seq, i, dropout):
    """Set p on a Dropout that already feeds seq[i]. True if there was one."""
    if i > 0 and isinstance(seq[i - 1], nn.Dropout):
        seq[i - 1].p = dropout
        return True
    return False


def build_model(name, num_classes, pretrained=True, dropout=0.5):
    """Build `name` with a fresh `num_classes` head, regularised by `dropout`."""
    weights = "IMAGENET1K_V1" if pretrained else None
    net = getattr(tvm, name)(weights=weights)

    # Four head layouts across torchvision. The final Linear is replaced in place;
    # nothing is ever inserted into a Sequential that already exists.
    if hasattr(net, "fc"):                                    # resnet
        net.fc = new_head(net.fc.in_features, num_classes, dropout)
    elif hasattr(net, "classifier"):                          # efficientnet, convnext,
        cls = net.classifier                                  # mobilenet, densenet
        if isinstance(cls, nn.Linear):                        # densenet
            net.classifier = new_head(cls.in_features, num_classes, dropout)
        else:
            i = max(j for j, m in enumerate(cls) if isinstance(m, nn.Linear))
            if retune_dropout_before(cls, i, dropout):        # efficientnet, mobilenet
                cls[i] = nn.Linear(cls[i].in_features, num_classes)
            else:                                             # convnext
                cls[i] = new_head(cls[i].in_features, num_classes, dropout)
    elif hasattr(net, "heads"):                               # torchvision ViT
        net.heads.head = new_head(net.heads.head.in_features, num_classes, dropout)
    elif hasattr(net, "head"):                                # swin
        net.head = new_head(net.head.in_features, num_classes, dropout)
    else:
        raise ValueError(f"{name}: head layout not recognised")
    return net


RESNET_STAGES = ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4"]


def freeze_until(net, stage):
    """Freeze every ResNet stage BEFORE `stage`."""
    if stage in (None, "none", ""):
        return param_counts(net)
    if stage not in RESNET_STAGES:
        raise ValueError(f"unknown stage {stage!r}. Known: {RESNET_STAGES}")
    if not all(hasattr(net, s) for s in RESNET_STAGES):
        raise ValueError("freeze_until only applies to ResNet-style backbones")
    for s in RESNET_STAGES[:RESNET_STAGES.index(stage)]:
        for p in getattr(net, s).parameters():
            p.requires_grad_(False)
    return param_counts(net)


def freeze_batchnorm(net):
    """Put every BatchNorm2d in eval() and freeze its affine parameters.

    Two separate things and both matter. Freezing the parameters stops gamma and
    beta adapting to the batch; eval() stops the running mean and variance moving.
    Must be reapplied every epoch, because model.train() puts everything back.
    """
    n = 0
    for m in net.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad_(False)
            n += 1
    return n


def set_backbone_eval(net):
    """Re-apply BatchNorm eval() after model.train()."""
    for m in net.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


def param_counts(net):
    total = sum(p.numel() for p in net.parameters())
    trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "frozen": total - trainable}


def apply_mps_workaround(model, device):
    """Force every BatchNorm input contiguous on the Apple GPU. Returns how many.

    A no-op on CUDA and CPU, so it can be called unconditionally. On MPS it
    sidesteps a backward bug that a non-contiguous BatchNorm input can trigger.
    Belt and braces on current torch, and cheap.
    """
    if device.type != "mps":
        return 0
    n = 0
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.register_forward_pre_hook(lambda m, i: (i[0].contiguous(),))
            n += 1
    return n


model = build_model(MODEL, NUM_CLASSES, pretrained=PRETRAINED, dropout=DROPOUT).to(DEVICE)
hooked = apply_mps_workaround(model, DEVICE)
if hooked:
    print(f"MPS: BatchNorm contiguity hook on {hooked} layers")
if FREEZE_UNTIL not in ("none", "", None):
    freeze_until(model, FREEZE_UNTIL)
if FREEZE_BN:
    freeze_batchnorm(model)

PARAMS = param_counts(model)
print(f"{MODEL}: {PARAMS['total']:,} parameters, {PARAMS['trainable']:,} trainable "
      f"({100 * PARAMS['trainable'] / PARAMS['total']:.2f}%), {PARAMS['frozen']:,} frozen")
print("\nnote: freezing to layer3 removes only 6.1% of the parameters, because the")
print("weight is almost all in layer4. What it bought was reproducibility — the seed")
print("spread fell from 0.026 to 0.003 — not a reduction in overfitting.")

## Loss, optimiser and schedule

Nothing surprising here. The one thing to note is that the optimiser is given only
the parameters that still require gradients, so the frozen stages are not merely
zeroed, they are absent from the optimiser state.

In [ ]:
weight = (class_weights(frames["train"], NUM_CLASSES, DEVICE)
          if CLASS_WEIGHTED_LOSS else None)
if weight is not None:
    print(f"class weights (patient level): {[round(w, 3) for w in weight.tolist()]}")

criterion = nn.CrossEntropyLoss(weight=weight, label_smoothing=LABEL_SMOOTHING)


def build_optimizer(model):
    params = [p for p in model.parameters() if p.requires_grad]
    if OPTIMIZER == "adamw":
        return torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    if OPTIMIZER == "sgd":
        return torch.optim.SGD(params, lr=LEARNING_RATE, momentum=0.9,
                               weight_decay=WEIGHT_DECAY)
    raise ValueError(f"unknown optimizer {OPTIMIZER!r}")


def build_scheduler(optimizer):
    if SCHEDULER == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    if SCHEDULER == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                          patience=5, factor=0.5)
    return None


optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print(f"{OPTIMIZER} lr={LEARNING_RATE} wd={WEIGHT_DECAY}, schedule {SCHEDULER}, "
      f"label smoothing {LABEL_SMOOTHING}")

## Metrics

The one rule this section enforces: everything reported is per patient. Slice
probabilities are aggregated into one prediction per patient before any metric is
computed.

Slices from one patient are near-duplicates, so a slice-level score measures how
well the model recognises the patient, not the disease. Slice-level numbers are
still computed because they are a useful overfitting signal, but they are labelled
as such and never used to select a model.

The headline metric is macro one-vs-rest AUC. It weights classes equally
regardless of prevalence and it is threshold-free. Everything this project measured
about threshold selection on small validation sets says a tuned threshold does not
transfer, so predictions are plain argmax and no threshold is ever fitted.

Two defensive behaviours are deliberate. A diverged run producing NaN logits gets
NaN metrics rather than a crash, so the training log shows which epoch went wrong
instead of a stack trace with no context. And a split missing a whole class gets a
NaN macro-AUC rather than a number computed over a subset, which would be silently
incomparable with every other number in the table.

In [ ]:
@torch.no_grad()
def predict(model, loader, device, use_amp=False):
    """Softmax probabilities, labels and row positions for a whole loader."""
    model.eval()
    probs, labels, idxs = [], [], []
    for x, y, i in loader:
        # Asynchronous copies are only safe from pinned memory, which this project
        # allocates on CUDA alone. See the training loop for the measurement.
        non_blocking = device.type == "cuda"
        x = x.to(device, non_blocking=non_blocking)
        with torch.autocast(device.type, enabled=use_amp and device.type == "cuda"):
            logits = model(x)
        probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        labels.append(y.numpy())
        idxs.append(i.numpy())
    return np.concatenate(probs), np.concatenate(labels), np.concatenate(idxs)


def aggregate_by_patient(probs, idxs, rows, how="mean"):
    """Collapse slice probabilities into one prediction per patient."""
    # Explicit column list, not a startswith("p") filter — that would also match
    # `pid` and cheerfully try to average patient identifiers.
    prob_cols = [f"class_{c}" for c in range(probs.shape[1])]
    df = pd.DataFrame(probs, columns=prob_cols)
    df["pid"] = rows.iloc[idxs].pid.values
    df["label"] = rows.iloc[idxs].label.values

    agg = df.groupby("pid").agg({**{c: how for c in prob_cols}, "label": "first"})
    p = agg[prob_cols].to_numpy()
    # Renormalise: a median over slices does not sum to 1.
    p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-9, None)
    return p, agg["label"].to_numpy(), agg.index.tolist()


def compute_metrics(y_true, y_prob, class_names):
    """Every metric this project reports, from labels and probabilities."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=float)
    n = y_prob.shape[1]
    present = np.unique(y_true)

    finite = np.isfinite(y_prob).all()
    if not finite:
        y_prob = np.nan_to_num(y_prob, nan=1.0 / n, posinf=1.0, neginf=0.0)
    y_pred = y_prob.argmax(1)

    if len(present) == n and finite:
        macro_auc = float(roc_auc_score(y_true, y_prob[:, 1]) if n == 2 else
                          roc_auc_score(y_true, y_prob, multi_class="ovr",
                                        average="macro"))
        per_class_auc = [float(roc_auc_score((y_true == c).astype(int), y_prob[:, c]))
                         for c in range(n)]
    else:
        macro_auc = float("nan")
        per_class_auc = [float("nan")] * n

    return {
        "n_patients": int(len(y_true)),
        "auc": macro_auc,                       # the headline metric
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro",
                                                 zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro",
                                           zero_division=0)),
        "per_class_auc": [round(a, 4) for a in per_class_auc],
        "per_class_precision": [round(float(v), 4) for v in
                                precision_score(y_true, y_pred, average=None,
                                                labels=list(range(n)), zero_division=0)],
        "per_class_recall": [round(float(v), 4) for v in
                             recall_score(y_true, y_pred, average=None,
                                          labels=list(range(n)), zero_division=0)],
        "per_class_f1": [round(float(v), 4) for v in
                         f1_score(y_true, y_pred, average=None,
                                  labels=list(range(n)), zero_division=0)],
        "class_counts": [int((y_true == c).sum()) for c in range(n)],
        # Rows = truth, columns = prediction. Two runs with the same macro-AUC can
        # predict entirely different classes; without this that stays invisible.
        "confusion": confusion_matrix(y_true, y_pred, labels=list(range(n))).tolist(),
        # Accuracy means nothing without this beside it.
        "trivial_baseline_accuracy": float(max((y_true == c).mean() for c in range(n))),
    }


def predictions_frame(pids, y_true, y_prob, class_names, rows=None):
    """One row per patient: truth, prediction, and every class probability."""
    df = pd.DataFrame({"pid": pids, "label": y_true,
                       "label_name": [class_names[i] for i in y_true],
                       "pred": y_prob.argmax(1),
                       "pred_name": [class_names[i] for i in y_prob.argmax(1)]})
    for c, name in enumerate(class_names):
        df[f"prob_{name}"] = y_prob[:, c]
    df["correct"] = df.label == df.pred
    # Cohort travels with the prediction, so a per-cohort AUC needs no second join.
    if rows is not None and "cohort" in rows.columns:
        df = df.merge(rows.drop_duplicates("pid")[["pid", "cohort"]], on="pid", how="left")
    return df


def evaluate_split(model, loader, rows, device, use_amp):
    """Patient-level metrics for one split, plus what is needed to plot them."""
    probs, labels, idxs = predict(model, loader, device, use_amp)
    p, y, pids = aggregate_by_patient(probs, idxs, rows, AGGREGATION)
    metrics = compute_metrics(y, p, CLASS_NAMES)
    # Slice-level numbers alongside, purely as an overfitting signal.
    metrics["slice_accuracy"] = float((probs.argmax(1) == labels).mean())
    eps = 1e-9
    metrics["slice_loss"] = float(
        -np.log(np.clip(probs[np.arange(len(labels)), labels], eps, 1.0)).mean())
    return metrics, y, p, pids


print("metrics defined — everything patient level, macro OVR AUC is the headline")

## Figures

Written automatically at the end of the run, not by hand. A run that needs a human
afterwards is a run that will be reported inconsistently.

The AUC and the generalisation gap share a figure on purpose. A change that lowers
training accuracy without lowering test AUC has reduced overfitting, and that is
invisible if the two are plotted apart.

The precision-recall curves carry a dotted line at the class prevalence, because
the baseline for a PR curve is the prevalence and not 0.5. A PR curve without it
cannot be read.

In [ ]:
def roc_points(y_true, y_prob, n_classes):
    out = {}
    for c in range(n_classes):
        binary = (np.asarray(y_true) == c).astype(int)
        if binary.sum() in (0, len(binary)):
            continue
        fpr, tpr, _ = roc_curve(binary, y_prob[:, c])
        out[c] = (fpr, tpr, float(roc_auc_score(binary, y_prob[:, c])))
    return out


def pr_points(y_true, y_prob, n_classes):
    out = {}
    for c in range(n_classes):
        binary = (np.asarray(y_true) == c).astype(int)
        if binary.sum() in (0, len(binary)):
            continue
        precision, recall, _ = precision_recall_curve(binary, y_prob[:, c])
        out[c] = (recall, precision, float(binary.mean()))
    return out


def plot_curves(history, out_dir, best_epoch=None):
    """Loss, accuracy, and the AUC beside the generalisation gap."""
    def mark(ax):
        if best_epoch:
            ax.axvline(best_epoch, color="k", ls="--", lw=1, alpha=0.6)
            ax.text(best_epoch, ax.get_ylim()[1], f" best ep {best_epoch}",
                    fontsize=7, va="top")

    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(history.epoch, history.train_loss, label="train", color=PALETTE[0])
    ax.plot(history.epoch, history.val_loss, label="validation", color=PALETTE[1])
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("Loss")
    ax.legend(); mark(ax)
    fig.savefig(out_dir / "loss_curve.png", bbox_inches="tight"); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(history.epoch, history.train_acc, label="train (slice)", color=PALETTE[0])
    ax.plot(history.epoch, history.val_accuracy, label="validation (patient)",
            color=PALETTE[1])
    ax.plot(history.epoch, history.val_balanced_accuracy,
            label="validation balanced", color=PALETTE[2], ls="--")
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.set_title("Accuracy")
    ax.legend(fontsize=7); mark(ax)
    fig.savefig(out_dir / "accuracy_curve.png", bbox_inches="tight"); plt.close(fig)

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(history.epoch, history.val_auc, color=PALETTE[1])
    ax[0].axhline(0.5, color="grey", ls=":", lw=1)
    ax[0].set_title("Validation macro-AUC (patient level)")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("macro-AUC"); mark(ax[0])
    gap = history.train_acc - history.val_accuracy
    ax[1].plot(history.epoch, gap, color=PALETTE[3])
    ax[1].axhline(0, color="grey", ls=":", lw=1)
    ax[1].set_title("Generalisation gap  (train acc − val acc)")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("gap"); mark(ax[1])
    fig.savefig(out_dir / "auc_and_gap_curve.png", bbox_inches="tight"); plt.close(fig)


def plot_confusion(cm, class_names, out_path, title="Confusion matrix"):
    """Counts and row-normalised percentages in one figure."""
    cm = np.asarray(cm, dtype=float)
    pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1e-9, None)
    fig, axes = plt.subplots(1, 2, figsize=(4.2 * len(class_names), 4.2))
    for ax, data, fmt, lab in ((axes[0], cm, "{:.0f}", "count"),
                               (axes[1], pct, "{:.2f}", "row-normalised")):
        im = ax.imshow(data, cmap="Blues", vmin=0)
        ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha="right")
        ax.set_yticks(range(len(class_names)), class_names)
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
        ax.set_title(f"{title} — {lab}", fontsize=9); ax.grid(False)
        thr = data.max() / 2 if data.max() else 0.5
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                ax.text(j, i, fmt.format(data[i, j]), ha="center", va="center",
                        color="white" if data[i, j] > thr else "black", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_roc(y_true, y_prob, class_names, out_path):
    curves = roc_points(y_true, y_prob, len(class_names))
    fig, ax = plt.subplots(figsize=(5, 4.6))
    for c, (fpr, tpr, auc) in curves.items():
        ax.plot(fpr, tpr, color=PALETTE[c % len(PALETTE)],
                label=f"{class_names[c]}  AUC {auc:.3f}")
    ax.plot([0, 1], [0, 1], color="grey", ls=":", lw=1, label="chance")
    ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
    ax.set_title("ROC — one vs rest, patient level")
    ax.legend(fontsize=8, loc="lower right")
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_pr(y_true, y_prob, class_names, out_path):
    curves = pr_points(y_true, y_prob, len(class_names))
    fig, ax = plt.subplots(figsize=(5, 4.6))
    for c, (recall, precision, base) in curves.items():
        col = PALETTE[c % len(PALETTE)]
        ax.plot(recall, precision, color=col, label=f"{class_names[c]}")
        ax.axhline(base, color=col, ls=":", lw=1, alpha=0.7)
    ax.set_xlabel("recall"); ax.set_ylabel("precision")
    ax.set_title("Precision–recall (dotted = class prevalence)")
    ax.legend(fontsize=8, loc="lower left")
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


def plot_per_class(metrics, class_names, out_path):
    keys = ["per_class_auc", "per_class_precision", "per_class_recall", "per_class_f1"]
    labels = ["AUC", "precision", "recall", "F1"]
    data = np.array([metrics[k] for k in keys], dtype=float)
    x = np.arange(len(class_names)); width = 0.2
    fig, ax = plt.subplots(figsize=(1.9 * len(class_names) + 3, 3.8))
    for i, (row, lab) in enumerate(zip(data, labels)):
        ax.bar(x + (i - 1.5) * width, row, width, label=lab,
               color=PALETTE[i % len(PALETTE)])
    ax.axhline(0.5, color="grey", ls=":", lw=1)
    ax.set_xticks(x, class_names, rotation=20, ha="right")
    ax.set_ylim(0, 1); ax.set_ylabel("score")
    ax.set_title("Per-class metrics, patient level"); ax.legend(fontsize=8, ncol=4)
    fig.savefig(out_path, bbox_inches="tight"); plt.close(fig)


print("figure writers defined")

## The run folder

Numbered `test_NNN_<model>_<task>`, one above the highest already present. Gaps are
never reused, so a deleted experiment cannot have its number silently taken by a
later one.

The folder and its `config.json` are written before training starts, not after. A
run that dies at epoch 3 must still say what it was trying to do, and that is how
you tell it apart from a run that never began.

Note that the numbering scans `results/classifier/` for existing `test_NNN_`
folders. If you move old runs out of that folder, the counter restarts. Check what
this cell prints before it trains for an hour.

In [ ]:
import re

RUN_PATTERN = re.compile(r"^test_(\d+)_")


def next_number(results_dir):
    """One above the highest number already present."""
    results_dir.mkdir(parents=True, exist_ok=True)
    used = [int(m.group(1)) for d in results_dir.iterdir() if d.is_dir()
            for m in [RUN_PATTERN.match(d.name)] if m]
    return max(used, default=0) + 1


def new_experiment(results_dir, config, model_name, task, suffix=""):
    """Create and return results/classifier/test_NNN_<model>_<task>/."""
    n = next_number(results_dir)
    parts = [f"test_{n:03d}", model_name, task]
    if suffix:
        parts.append(suffix)
    run_dir = results_dir / "_".join(parts)
    (run_dir / "figures").mkdir(parents=True, exist_ok=True)
    (run_dir / "checkpoints").mkdir(parents=True, exist_ok=True)

    # Written FIRST, not last.
    (run_dir / "config.json").write_text(json.dumps(config, indent=2, default=str))
    (run_dir / "README.md").write_text(
        f"# {run_dir.name}\n\n"
        f"task **{task}** · model **{model_name}** · seed **{config['seed']}**\n\n"
        f"Generated by notebook 03. Full configuration in `config.json`, "
        f"metrics in `metrics.csv`, figures in `figures/`.\n")
    return run_dir


RUN_DIR = new_experiment(RESULTS_ROOT, CONFIG, MODEL, TASK)
print(f"run folder: {RUN_DIR}")
print(f"  checkpoints/  best_model.pt, last_model.pt")
print(f"  figures/      written at the end")

## One training epoch

This is the function where three separate defects lived, all of them presenting as
"MPS is broken". Each guard below is keyed to the measurement that found it, and
none of them changes anything on CUDA.

**The asynchronous copy.** `non_blocking=True` is only safe when the source is
pinned memory, and this project pins on CUDA alone. On MPS the copy returned before
it had finished while the DataLoader was free to reuse the source buffer, so the
model trained on partially overwritten batches. Measured over one full epoch, 508
steps, same seed and data: `non_blocking=True` on MPS gave loss NaN and train
accuracy 0.9425, blocking copies gave 1.1539 and 0.4372, and CPU gave 1.1502 and
0.4237.

**The disabled scaler.** A `GradScaler(enabled=False)` is documented as a no-op and
behaves like one on CPU. On MPS, routing the backward pass through it corrupts the
gradients. Same measurement: through the disabled scaler, loss NaN and accuracy
0.9000; plain backward, 1.1539 and 0.4372.

**The device-side accumulator.** Keeping the running totals as 0-dim device tensors
is a real speed win on CUDA, because reading `.item()` every batch costs about 20%
of the step. On MPS it silently produces garbage. So: device tensors on CUDA,
Python floats everywhere else.

There is also an explicit `torch.mps.synchronize()` per step. Without one the queue
runs far enough ahead that weights are corrupted to NaN partway through an epoch,
non-deterministically. CUDA keeps the fast path and MPS pays for correctness.

The progress bar shows a batch counter and deliberately nothing else. Showing a
running loss would mean reading `.item()` every step, which is exactly the
synchronisation this loop removed.

In [ ]:
from contextlib import nullcontext


def train_one_epoch(model, loader, criterion, optimizer, scaler, device,
                    use_amp, freeze_bn=False, progress=None):
    """One pass over the training set. Returns (mean loss, slice accuracy)."""
    model.train()
    if freeze_bn:
        set_backbone_eval(model)

    # Apple MPS needs an explicit synchronisation once per step. See the markdown.
    sync = torch.mps.synchronize if device.type == "mps" else None

    # Device tensors on CUDA, Python floats everywhere else. See the markdown.
    on_device = device.type == "cuda"
    running = torch.zeros((), device=device, dtype=torch.float32) if on_device else 0.0
    correct = torch.zeros((), device=device, dtype=torch.float32) if on_device else 0.0
    seen = 0

    steps = loader
    bar = None
    if progress:
        try:
            from tqdm.auto import tqdm
            bar = tqdm(loader, desc=progress, unit="batch", leave=False)
            steps = bar
        except ImportError:
            pass                  # tqdm is optional; training does not need it

    for x, y, _ in steps:
        # Asynchronous where it helps and is safe, synchronous everywhere else.
        non_blocking = device.type == "cuda"
        x = x.to(device, non_blocking=non_blocking)
        y = y.to(device, non_blocking=non_blocking)
        optimizer.zero_grad(set_to_none=True)

        # autocast is entered ONLY when AMP is actually on, which is CUDA only.
        with (torch.autocast(device.type, enabled=True) if use_amp else nullcontext()):
            logits = model(x)
            loss = criterion(logits, y)

        # The scaler is only touched when it is actually enabled.
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        if on_device:
            running += loss.detach() * y.size(0)
            correct += (logits.detach().argmax(1) == y).sum()
        else:
            running += loss.item() * y.size(0)
            correct += float((logits.detach().argmax(1) == y).sum().item())
        seen += y.size(0)
        if sync is not None:
            sync()

    if bar is not None:
        bar.close()

    total_loss = float(running.item()) if on_device else float(running)
    total_ok = float(correct.item()) if on_device else float(correct)
    return total_loss / seen, total_ok / seen


print("training loop defined")

## Train

Thirty epochs. Each one trains, then evaluates on validation at the patient level,
then decides whether to keep the checkpoint.

Two things about checkpoint selection. It is driven by the validation metric, never
by a training metric: a previous iteration of this project selected on training
accuracy and therefore picked whichever model let the network memorise its data
best, at 99% and climbing. And the learning rate is read before the scheduler
steps, so the logged value is the one that epoch actually trained with.

The `improved or epoch == 1` condition is there so a run that never improves still
has a checkpoint to evaluate at the end.

On a rented RTX A4500 this takes about 12 minutes. On an M-series Mac with the
image cache on, closer to two hours.

In [ ]:
history = []
best_score, best_epoch, stale = -np.inf, 0, 0
train_acc_at_best = float("nan")
sampler = getattr(loaders["train"], "batch_sampler", None)

print(f"training {MODEL} for {EPOCHS} epochs on {describe_device(DEVICE)}")
print("-" * 78)

for epoch in range(1, EPOCHS + 1):
    if hasattr(sampler, "set_epoch"):
        sampler.set_epoch(epoch)
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, loaders["train"], criterion, optimizer, scaler, DEVICE, USE_AMP,
        FREEZE_BN, progress=f"epoch {epoch:03d}/{EPOCHS:03d} training")

    val_metrics, _, _, _ = evaluate_split(model, loaders["val"], frames["val"],
                                          DEVICE, USE_AMP)

    # Read the rate BEFORE stepping, so the logged value is the one this epoch used.
    lr_now = optimizer.param_groups[-1]["lr"]
    if scheduler is not None:
        if SCHEDULER == "plateau":
            scheduler.step(val_metrics[MONITOR_METRIC])
        else:
            scheduler.step()

    history.append({
        "epoch": epoch, "lr": lr_now, "train_loss": train_loss,
        "train_acc": train_acc, "val_loss": val_metrics["slice_loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_auc": val_metrics["auc"], "val_macro_f1": val_metrics["macro_f1"],
        "seconds": time.time() - t0,
    })
    print(f"epoch {epoch:03d}/{EPOCHS:03d} done  lr {lr_now:.2e}  "
          f"loss {train_loss:.4f}  train_acc {train_acc:.4f} | "
          f"val AUC {val_metrics['auc']:.4f}  bal {val_metrics['balanced_accuracy']:.4f}"
          f"  ({time.time() - t0:.0f}s)")

    score = val_metrics[MONITOR_METRIC]
    improved = np.isfinite(score) and score > best_score + 1e-5
    if improved or epoch == 1:
        best_score, best_epoch, stale = score, epoch, 0
        train_acc_at_best = train_acc
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "config": CONFIG, "val_metrics": val_metrics},
                   RUN_DIR / "checkpoints" / "best_model.pt")
    else:
        stale += 1
    # `last_model.pt` every epoch, so a crash still leaves something to resume from.
    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "config": CONFIG, "val_metrics": val_metrics},
               RUN_DIR / "checkpoints" / "last_model.pt")

    if EARLY_STOPPING_PATIENCE > 0 and stale >= EARLY_STOPPING_PATIENCE:
        print(f"early stopping: {stale} epochs without improvement")
        break

print("-" * 78)
print(f"best {MONITOR_METRIC} {best_score:.4f} at epoch {best_epoch}")

## Final evaluation, from the selected checkpoint

Not from the model left in memory. The model in memory is whatever the last epoch
produced, which is a model that was never selected and is not what any reported
number describes.

`weights_only=False` is needed because the checkpoint carries the config dict
alongside the tensors.

In [ ]:
ck = torch.load(RUN_DIR / "checkpoints" / "best_model.pt", map_location=DEVICE,
                weights_only=False)
model.load_state_dict(ck["model_state_dict"])
print(f"loaded checkpoint from epoch {ck['epoch']}")

results = {}
for split in ("val", "test"):
    if split not in loaders:
        continue
    m, y, p, pids = evaluate_split(model, loaders[split], frames[split], DEVICE, USE_AMP)

    # Every artefact for this split.
    figures = RUN_DIR / "figures"
    plot_confusion(m["confusion"], CLASS_NAMES,
                   figures / f"confusion_matrix_{split}.png",
                   title=f"Confusion — {split}")
    plot_roc(y, p, CLASS_NAMES, figures / f"roc_curve_{split}.png")
    plot_pr(y, p, CLASS_NAMES, figures / f"pr_curve_{split}.png")
    plot_per_class(m, CLASS_NAMES, figures / f"per_class_{split}.png")

    predictions_frame(pids, y, p, CLASS_NAMES, frames[split]).to_csv(
        RUN_DIR / f"predictions_{split}.csv", index=False)
    (RUN_DIR / f"classification_report_{split}.txt").write_text(
        classification_report(y, p.argmax(1), labels=list(range(NUM_CLASSES)),
                              target_names=list(CLASS_NAMES), zero_division=0,
                              digits=4))

    results[split] = m
    print(f"{split:<5} macro-AUC {m['auc']:.4f} | acc {m['accuracy']:.4f} "
          f"(trivial {m['trivial_baseline_accuracy']:.4f}) | "
          f"balanced {m['balanced_accuracy']:.4f}")

## Write the run summary

The per-epoch log, the curves, the flat metrics table and `results.json`.

The generalisation gap uses the training accuracy at the selected epoch and not at
the last one, because the last epoch describes a model that was never used.

In [ ]:
hist = pd.DataFrame(history)
hist.to_csv(RUN_DIR / "train_log.csv", index=False)
plot_curves(hist, RUN_DIR / "figures", best_epoch)

# The three curve figures also go at the top level, where a reader looks first.
for name in ("loss_curve.png", "accuracy_curve.png", "auc_and_gap_curve.png"):
    src = RUN_DIR / "figures" / name
    if src.exists():
        (RUN_DIR / name).write_bytes(src.read_bytes())

# One flat row per split. This is the file notebook 05 reads.
flat = []
for split, m in results.items():
    row = {"split": split}
    for k, v in m.items():
        if isinstance(v, list) and k != "confusion":
            row.update({f"{k}_{i}": x for i, x in enumerate(v)})
        elif k != "confusion":
            row[k] = v
    flat.append(row)
pd.DataFrame(flat).to_csv(RUN_DIR / "metrics.csv", index=False)

(RUN_DIR / "results.json").write_text(json.dumps({
    "config": CONFIG, "best_epoch": best_epoch, "epochs_run": int(len(hist)),
    "parameters": PARAMS, "splits": results,
    "train_acc_at_best": train_acc_at_best,
    # Gap at the SELECTED epoch, not the last one.
    "generalisation_gap": {s: train_acc_at_best - results[s]["accuracy"]
                           for s in results},
}, indent=2, default=float))

print(f"written: {RUN_DIR}")
for f in sorted(RUN_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(RUN_DIR)}")

## Read the result honestly

The two lines this cell prints are the ones that decide whether the run means
anything. Accuracy above the trivial baseline says the model does something a
constant predictor does not. The generalisation gap says how much of the training
accuracy was memorisation.

And the noise floor applies to everything above. A difference of 0.05 macro-AUC
against another run of the same configuration is not a finding.

In [ ]:
t = results["test"]
print(f"macro AUC          {t['auc']:.4f}")
print(f"balanced accuracy  {t['balanced_accuracy']:.4f}")
print(f"accuracy           {t['accuracy']:.4f}")
print(f"trivial baseline   {t['trivial_baseline_accuracy']:.4f}")
print(f"above baseline     {t['accuracy'] - t['trivial_baseline_accuracy']:+.4f}")
print(f"train acc at best  {train_acc_at_best:.4f}")
print(f"generalisation gap {train_acc_at_best - t['accuracy']:+.4f}")
print()
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<14} AUC {t['per_class_auc'][i]:.4f}  "
          f"recall {t['per_class_recall'][i]:.4f}  "
          f"n = {t['class_counts'][i]}")
print()
print("One seed is not a result. The measured noise floor here is 0.067 macro-AUC.")

## What this notebook produced

Everything below is in the run folder printed above, `results/classifier/test_NNN_resnet18_subtype/`.

| path | what it is |
|---|---|
| `config.json` | the exact configuration, written before training started |
| `README.md` | one-line description of the run |
| `checkpoints/best_model.pt` | the selected model, chosen on validation macro-AUC |
| `checkpoints/last_model.pt` | the final epoch, whatever it was |
| `train_log.csv` | one row per epoch: loss, accuracies, validation AUC, seconds |
| `metrics.csv` | one flat row per split, the file notebook 05 reads |
| `results.json` | config, best epoch, parameter counts, every metric, the generalisation gap |
| `predictions_val.csv`, `predictions_test.csv` | one row per patient with every class probability and the cohort |
| `classification_report_val.txt`, `_test.txt` | sklearn's report, saved verbatim |
| `loss_curve.png`, `accuracy_curve.png`, `auc_and_gap_curve.png` | the three curves, also copied into `figures/` |
| `figures/confusion_matrix_*.png` | counts and row-normalised, per split |
| `figures/roc_curve_*.png`, `pr_curve_*.png`, `per_class_*.png` | per-split diagnostics |

**Where this goes next**

Notebook 04 reads one run folder and adds the analyses that need judgement:
per-cohort breakdown, the source probe and the mistakes worth looking at.
Notebook 05 compares several runs with the noise floor applied.

The federated arm in notebooks 06 and 07 uses this identical configuration on every
hospital. That is what makes the comparison a comparison of federation rather than
of hyperparameters.

**A note on which run to trust.** The reported centralised baseline of this thesis
is not produced here. It lives at `results/federated/test01_centralized/seed_42/`
with macro AUC 0.6067918080145278, and it was produced by the same logic driven
from the batch script on the same GPU as the federated runs. This notebook is the
interactive path, which defaults to writing both checkpoints and a full report, and
it is the right place to try a change before committing a campaign to it.